In [6]:
import pandas as pd
from datetime import datetime, timedelta

csv_file_path = "//10.69.168.1/crnldata/forgetting/Carla/Fed/WT/AHAD12.108/3 mois/AHAD12.108_11_03_2026au18_03_2026.CSV"

def transform_dates():
    try:
        print("Lecture du fichier...")
        df = pd.read_csv(csv_file_path, sep=',')
        
        date_column = df.columns[0]
        print(f"Exemple brut dans le fichier: {df[date_column].iloc[0]}")
        # → affiche "1/8/2001 0:06:59"  =  8 janvier 2001 (format US M/D/YYYY)

        # Référence ANCIENNE : première date du fichier, lue en format US
        old_reference = datetime.strptime("1/8/2001 0:06:59", "%m/%d/%Y %H:%M:%S")
        # = 8 janvier 2001

        # Référence NOUVELLE : la vraie date de l'expérience, vous la connaissez en format FR
        # Le fichier couvre du 11/03/2026 au 18/03/2026
        # La première mesure correspond au 11/03/2026 à 17:04:09
        new_reference = datetime.strptime("11/03/2026 17:04:09", "%d/%m/%Y %H:%M:%S")
        # = 11 mars 2026

        time_offset = new_reference - old_reference
        print(f"Décalage: {time_offset}")

        def transform_single_date(date_str):
            if pd.isna(date_str):
                return date_str
            try:
                date_str_clean = str(date_str).strip()
                # Le fichier est en format US : M/D/YYYY H:MM:SS
                parsed_date = datetime.strptime(date_str_clean, "%m/%d/%Y %H:%M:%S")
                new_date = parsed_date + time_offset
                # On ressort en format FR : DD/MM/YYYY HH:MM:SS
                return new_date.strftime("%#m/%#d/%Y %H:%M:%S")
            except Exception as e:
                print(f"Erreur sur '{date_str}': {e}")
                return date_str

        print("\nTransformation en cours...")
        df[date_column] = df[date_column].apply(transform_single_date)

        print(f"Exemple après transformation: {df[date_column].iloc[0]}")
        # → doit afficher "11/03/2026 17:04:09"
        print(df[[date_column]].head(5))

        output_path = csv_file_path.replace(".CSV", "_transformed.CSV")
        df.to_csv(output_path, index=False, sep=',')
        print(f"\n✓ Sauvegardé : {output_path}")

    except Exception as e:
        print(f"❌ Erreur: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    transform_dates()

Lecture du fichier...
Exemple brut dans le fichier: 1/8/2001 0:06:59
Décalage: 9193 days, 16:57:10

Transformation en cours...
Exemple après transformation: 3/11/2026 17:04:09
  MM:DD:YYYY hh:mm:ss
0  3/11/2026 17:04:09
1  3/11/2026 17:04:12
2  3/11/2026 17:35:51
3  3/11/2026 17:35:51
4  3/11/2026 17:39:03

✓ Sauvegardé : //10.69.168.1/crnldata/forgetting/Carla/Fed/WT/AHAD12.108/3 mois/AHAD12.108_11_03_2026au18_03_2026_transformed.CSV
